In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  
EVENT_RAW_PATH = PROJECT_ROOT / "data" / "event_data" / "raw" / "event_data.json"
EVENT_CLEAN_DIR = PROJECT_ROOT / "data" / "event_data" / "clean"
EVENT_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

with open(EVENT_RAW_PATH, "r") as f:
    matches = json.load(f)

print(f"Loaded {len(matches)} matches")

Loaded 9999 matches


In [2]:
deaths_records = []

for match in matches:
    match_id = match["match_id"]
    for player in match.get("players", []):
        account_id = player.get("account_id")
        team = player.get("team")
        hero_id = player.get("hero_id")
        player_slot = player.get("player_slot")

        for death in player.get("death_details", []):
            deaths_records.append({
                "match_id": match_id,
                "account_id": account_id,
                "team": team,
                "hero_id": hero_id,
                "player_slot": player_slot,
                "game_time_s": death.get("game_time_s"),
                "death_duration_s": death.get("death_duration_s"),
                "killer_player_slot": death.get("killer_player_slot"),
                "time_to_kill_s": death.get("time_to_kill_s"),
                "death_pos_x": death["death_pos"][0] if death.get("death_pos") else None,
                "death_pos_y": death["death_pos"][1] if death.get("death_pos") else None,
                "death_pos_z": death["death_pos"][2] if death.get("death_pos") else None,
            })

deaths_df = pd.DataFrame(deaths_records)
print(f"Deaths table: {deaths_df.shape[0]} rows, {deaths_df.shape[1]} columns")
deaths_df.head()

Deaths table: 845491 rows, 12 columns


,match_id,account_id,team,hero_id,player_slot,game_time_s,death_duration_s,killer_player_slot,time_to_kill_s,death_pos_x,death_pos_y,death_pos_z
0,25200342,1271265534,Team0,7,2,278,8,7,None,-2748.2188,-661.62500,248.03125
1,25200342,1271265534,Team0,7,2,385,11,8,None,-2645.0625,-640.34375,256.40625
2,25200342,1271265534,Team0,7,2,588,18,7,None,-1442.4062,1653.12500,196.68750
3,25200342,1271265534,Team0,7,2,695,22,8,None,-2034.4062,-3419.96880,384.06250
4,25200342,1271265534,Team0,7,2,821,27,8,None,-2694.4062,-796.50000,258.28125


In [3]:
objectives_records = []

for match in matches:
    match_id = match["match_id"]
    for obj in match.get("objectives", []):
        objectives_records.append({
            "match_id": match_id,
            "team_objective": obj.get("team_objective"),
            "team": obj.get("team"),
            "destroyed_time_s": obj.get("destroyed_time_s"),
            "first_damage_time_s": obj.get("first_damage_time_s"),
            "creep_damage": obj.get("creep_damage"),
            "player_damage": obj.get("player_damage"),
        })

objectives_df = pd.DataFrame(objectives_records)
print(f"Objectives table: {objectives_df.shape[0]} rows, {objectives_df.shape[1]} columns")
objectives_df.head()

Objectives table: 265547 rows, 7 columns


,match_id,team_objective,team,destroyed_time_s,first_damage_time_s,creep_damage,player_damage
0,25200342,Tier1Lane3,Team0,780,53,2595,2689
1,25200342,Tier1Lane4,Team0,501,55,5152,354
2,25200342,Tier1Lane2,Team1,808,59,2739,2253
3,25200342,Tier2Lane4,Team0,907,90,2513,4299
4,25200342,Tier1Lane2,Team0,620,179,2340,2246


In [4]:
mid_boss_records = []

for match in matches:
    match_id = match["match_id"]
    for mb in match.get("mid_boss", []):
        mid_boss_records.append({
            "match_id": match_id,
            "team_killed": mb.get("team_killed"),
            "team_claimed": mb.get("team_claimed"),
            "destroyed_time_s": mb.get("destroyed_time_s"),
        })

mid_boss_df = pd.DataFrame(mid_boss_records)
print(f"Mid boss table: {mid_boss_df.shape[0]} rows, {mid_boss_df.shape[1]} columns")
mid_boss_df.head()

Mid boss table: 9964 rows, 4 columns


,match_id,team_killed,team_claimed,destroyed_time_s
0,25200342,Team0,Team0,1302
1,25200372,Team0,Team0,1857
2,25200377,Team1,Team1,1631
3,25200377,Team1,Team1,2086
4,25200377,Team1,Team0,2494


In [5]:
deaths_df.to_parquet(EVENT_CLEAN_DIR / "deaths_clean.parquet")
objectives_df.to_parquet(EVENT_CLEAN_DIR / "objectives_clean.parquet")
mid_boss_df.to_parquet(EVENT_CLEAN_DIR / "mid_boss_clean.parquet")

print("Saved:")
print(f"  {EVENT_CLEAN_DIR / 'deaths_clean.parquet'} — {deaths_df.shape[0]} rows")
print(f"  {EVENT_CLEAN_DIR / 'objectives_clean.parquet'} — {objectives_df.shape[0]} rows")
print(f"  {EVENT_CLEAN_DIR / 'mid_boss_clean.parquet'} — {mid_boss_df.shape[0]} rows")

Saved:
  C:\.STUFF\coding\dl-match-analyser\data\event_data\clean\deaths_clean.parquet — 845491 rows
  C:\.STUFF\coding\dl-match-analyser\data\event_data\clean\objectives_clean.parquet — 265547 rows
  C:\.STUFF\coding\dl-match-analyser\data\event_data\clean\mid_boss_clean.parquet — 9964 rows
